# Specifying demand tariffs using time periods
The example shows how to use time periods to define demand tariffs, and how to set a datetime series for the optimiser to use.

In [53]:
import numpy as np
np.set_printoptions(suppress=True)
from echo.echo_models import *
from echo.echo_optimiser import *
from echo.objectives import *
import pprint


## The first thing we need to do is define a profile for the optimiser, so that it knows what dates/times the intervals correspond to.
Here we simulate a week in Jan 2021, in 15 min intervals

In [54]:
profile = pd.DataFrame(
    index=pd.date_range(start=datetime(2021, 1, 4), end=datetime(2021, 1, 11), freq='15min', closed='left'))

time_periods = len(profile)
print(time_periods)
interval_duration = 15

672


C:\Users\61405\AppData\Local\Temp\ipykernel_11436\2987873827.py:2: FutureWarning: Argument `closed` is deprecated in favor of `inclusive`.
  index=pd.date_range(start=datetime(2021, 1, 4), end=datetime(2021, 1, 11), freq='15min', closed='left'))


## Build a basic btm site (see btm_battery_example for more details)

In [55]:
expansion_periods = 1  # Number of planning intervals - in echo V1, set to 1 always
num_days = 7
test_load = np.array(
    [2.13, 2.09, 2.3, 2.11, 2.2, 2.23, 2.2, 2.15, 2.02, 2.19, 2.19, 2.19, 2.12, 2.15, 2.25, 2.12, 2.21, 2.16,
     2.26, 2.13, 2.08, 2.15, 2.42, 2.02, 2.3, 2.26, 2.35, 2.55, 3.23, 2.98, 3.49, 3.5, 3.12, 3.52, 3.94, 3.55,
     3.99, 3.71, 3.38, 3.76, 3.71, 3.78, 3.29, 3.65, 3.61, 3.75, 3.38, 3.66, 3.56, 3.69, 3.3, 3.61, 3.71, 3.82,
     3.17, 3.69, 3.74, 3.86, 3.57, 3.55, 3.75, 3.6, 3.67, 3.48, 3.51, 3.46, 3.19, 3.38, 3.19, 3.38, 3.04, 3.12,
     2.91, 3.11, 3.13, 2.77, 2.24, 2.54, 2.24, 2.24, 2.09, 2.33, 2.17, 2.16, 1.97, 2.16, 2.21, 2.18, 2.01, 2.16,
     2.19, 2.11, 2.17, 2.13, 2.05, 2.19]*num_days)
print(len(test_load))
test_pv = 2 * np.array(
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.05, 0.23, 0.52,
     0.74, 0.71, 0.63, 0.68, 0.97, 0.01, 0.52, 0.83, 0.83, 0.79, 1.22, 1.36, 1.27, 1.42, 1.97, 2.56, 2.91, 3.24,
     3.8, 4.3, 4.62, 4.84, 4.6, 4.17, 3.77, 3.76, 3.38, 2.64, 1.96, 1.76, 1.85, 2.4, 3.82, 5.13, 4.97, 5.02, 5.43,
     5.32, 3.56, 1.75, 1.43, 1.65, 1.69, 2.3, 2.71, 2.41, 2.63, 2.6, 1.9, 0.78, 0.13, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
     0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]*num_days)
test_pv *= -1  # convert solar generation to negative to match convention.

system = OptimisationGraph()

# Create assets
grid = Node()
grid.add_electrical_ports_from_list(['grid'])

connection_point = TellegenNode()
connection_point.add_electrical_ports_from_list(['load', 'battery', 'pv', 'grid'])

load = Node()
l1 = ElectricalDemand()
l1.add_demand_profile_from_array(test_load, expansion_periods)
load.ports['load'] = l1

battery = Node()
b = ElectricalStorage(max_capacity=15.0,
                       depth_of_discharge_limit=0,
                       charging_power_limit=1.25,
                       discharging_power_limit=-1.25,
                       charging_efficiency=1,
                       discharging_efficiency=1,
                       initial_state_of_charge=0.0)
battery.ports['battery'] = b

solar = Node()
pv = ElectricalGeneration()
pv.curtailable = False
pv.add_generation_profile_from_array(test_pv, expansion_periods)
solar.ports['pv'] = pv

# Populate graph with assets (nodes)
system.add_node_obj([grid, battery, load, solar, connection_point])

# Add edges to graph
system.connect_ports_and_create_edge(grid.ports['grid'], connection_point.ports['grid'])
system.connect_ports_and_create_edge(connection_point.ports['load'], load.ports['load'])
system.connect_ports_and_create_edge(connection_point.ports['battery'], battery.ports['battery'])
system.connect_ports_and_create_edge(connection_point.ports['pv'], solar.ports['pv'])

672


## Create a demand tariff
A demand tariff is an object that contains a list of demand charges.
Each demand charge specifies a rate and a window where the demand charge applies.
There is also an option to specify how often the demand charge calculation resets.
The method of specifying the reset depends on how the demand charge window is specified.

Demand charges apply to instantaneous power, so we can specify that the charge applies to either imports or exports.


# First we need to define a rate and window for the demand charge
We use the Window class to define the times when the charge applies.
A Window contains a list of time periods, as well as an optional reset period argument.
The time periods are defined using a start time, end time, and day type (list)

In [56]:
# create an import demand tariff
# peak usage
peak_rate = 2.0  # the rate is applied per kW

peak_window = Window(
    time_periods=[
        TimePeriod(start_time=time(7,0), end_time=time(9,0), day_type=[Day.weekday, Day.weekday]),
        TimePeriod(start_time=time(18, 0),end_time=time(21, 0),day_type=[Day.weekday, Day.weekend])
    ],
    reset_periods=ResetPeriod.day
)

### Now we can define the demand charge using the ImportDemandCharge class
We need to use ImportDemandCharge rather than DemandCharge because we want the charge to apply to imports (positive values) only.

In [57]:
# Now we can specify the demand charge, noting that we need to specify that it is an import charge
# We also need to use the window_object argument, not the window_array.
peak_charge = ImportDemandCharge(rate=peak_rate,
                           window_object=peak_window,
                           min_demand=0.0)

demand_tariff = DemandTariffObjective(component=connection_point.ports['grid'],
                                      demand_charges=[peak_charge])

objective_set = ObjectiveSet(objective_list=[demand_tariff])

## Run optimiser

In [58]:
optimiser = EchoOptimiser(
    interval_duration=interval_duration,
    number_of_intervals=time_periods,
    number_of_expansion_intervals=1,
    discount_rate=0,
    ES=system,
    objective_set=objective_set,
    profile=profile
)

optimiser.optimise()


C:\Users\61405\Documents\echo\echo\objectives.py:378: FutureWarning: `include_start` and `include_end` are deprecated in favour of `inclusive`.
  df.between_time(self.start_time, self.end_time, include_start=True, include_end=False).index))


## Check the results
We can check the peak demand value by looking at the .max_demand_val attribute of the demand charge object.
Note that this will return a list of values, one for each period where the calculation is reset.
Because we are simulating a week, and resetting our calculation every day, we will have 7 values.

In [59]:
print(optimiser.values(peak_charge.max_demand_val,0))

[1.88 1.88 1.88 1.88 0.   0.   0.  ]
